<a href="https://colab.research.google.com/github/dev-ago-uy/ExamenM-duloII.DM/blob/main/launcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import json
import requests
from datetime import datetime

API_KEY = os.getenv("SKYSCANNER_API_KEY")

CREATE_URL = "https://partners.api.skyscanner.net/apiservices/v3/flights/live/search/create"
POLL_URL = "https://partners.api.skyscanner.net/apiservices/v3/flights/live/search/poll/{session_token}"

ORIGIN = "MVD"        # Montevideo
DESTINATION = "MAD"   # Madrid
DATE_OUT = "2026-06-10"
DATE_BACK = "2026-06-25"
PRICE_ALERT_MAX = 850  # alerta si encuentra algo <= este valor
CURRENCY = "USD"
LOCALE = "es-ES"
MARKET = "UY"

def create_search():
    headers = {
        "x-api-key": API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "query": {
            "market": MARKET,
            "locale": LOCALE,
            "currency": CURRENCY,
            "queryLegs": [
                {
                    "originPlaceId": {"iata": ORIGIN},
                    "destinationPlaceId": {"iata": DESTINATION},
                    "date": {"year": int(DATE_OUT[:4]), "month": int(DATE_OUT[5:7]), "day": int(DATE_OUT[8:10])}
                },
                {
                    "originPlaceId": {"iata": DESTINATION},
                    "destinationPlaceId": {"iata": ORIGIN},
                    "date": {"year": int(DATE_BACK[:4]), "month": int(DATE_BACK[5:7]), "day": int(DATE_BACK[8:10])}
                }
            ],
            "adults": 1,
            "cabinClass": "CABIN_CLASS_ECONOMY"
        }
    }

    response = requests.post(CREATE_URL, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    return response.json()

def poll_search(session_token, max_attempts=8, sleep_seconds=5):
    headers = {
        "x-api-key": API_KEY,
        "Content-Type": "application/json"
    }

    url = POLL_URL.format(session_token=session_token)

    for attempt in range(max_attempts):
        response = requests.post(url, headers=headers, json={}, timeout=60)
        response.raise_for_status()
        data = response.json()

        status = data.get("status")
        if status == "RESULT_STATUS_COMPLETE":
            return data

        time.sleep(sleep_seconds)

    return data

def extract_cheapest(data):
    # Esta parte puede requerir ajuste fino según la estructura exacta devuelta por la API.
    itineraries = data.get("content", {}).get("results", {}).get("itineraries", {})
    if not itineraries:
        return None

    cheapest = None

    for itinerary_id, itinerary in itineraries.items():
        pricing_options = itinerary.get("pricingOptions", [])
        for option in pricing_options:
            price_info = option.get("price", {})
            amount = price_info.get("amount")
            if amount is None:
                continue

            try:
                amount = float(amount)
            except Exception:
                continue

            if cheapest is None or amount < cheapest["price"]:
                cheapest = {
                    "itinerary_id": itinerary_id,
                    "price": amount,
                    "agent_ids": option.get("agentIds", []),
                    "items": itinerary
                }

    return cheapest

def notify(result):
    # Podés cambiar esto por email, WhatsApp, Telegram o guardar en DB
    print("=== ALERTA DE VUELO BARATO ===")
    print(f"Ruta: {ORIGIN} -> {DESTINATION}")
    print(f"Fechas: {DATE_OUT} / {DATE_BACK}")
    print(f"Precio encontrado: {result['price']} {CURRENCY}")
    print(f"Fecha de búsqueda: {datetime.now().isoformat()}")

def main():
    if not API_KEY:
        raise RuntimeError("Falta SKYSCANNER_API_KEY en variables de entorno.")

    created = create_search()
    session_token = created.get("sessionToken")

    if not session_token:
        raise RuntimeError("No se recibió sessionToken en la respuesta inicial.")

    final_data = poll_search(session_token)
    cheapest = extract_cheapest(final_data)

    if not cheapest:
        print("No se encontraron resultados.")
        return

    print(f"Mejor precio encontrado: {cheapest['price']} {CURRENCY}")

    if cheapest["price"] <= PRICE_ALERT_MAX:
        notify(cheapest)

if __name__ == "__main__":
    main()

RuntimeError: Falta SKYSCANNER_API_KEY en variables de entorno.